<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/04_ra_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
# Install Unsloth and dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb
!pip install -U bitsandbytes>=0.46.1
!pip install --upgrade transformers
!pip install sentencepiece tiktoken

In [ ]:
import os, sys
import torch
from google.colab import drive
from google.colab import userdata
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)
from vector_store import RailVectorVault
from embed import RailEmbedder
from generator import RailDataGenerator

# Mount Drive to access Vector DB and save the model
drive.mount('/content/drive')
DB_PATH = "/content/drive/MyDrive/rail_ai_project/vector_db"
model_name = "unsloth/Llama-3.2-3B-Instruct"
adapter_path = "/content/drive/MyDrive/rail_ai_project/rail_safety_adapters"

In [ ]:
embedder = RailEmbedder(model_name='BAAI/bge-base-en-v1.5')
vault = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.for_inference(model)
model.load_adapter(adapter_path)

In [ ]:
def run_integrated_audit(questions, vault, tokenizer=tokenizer, model=model, method = 'rerank', n_results = 10, source_filter = None, show_context = False):

    # Methods: 'vector', 'hybrid', 'rerank'

    # We pull 10 chunks to give the model plenty of "Open Book" evidence
    # results = vault.query(question, n_results=n_results)
    # context = "\n---\n".join(results['documents'][0])
    if isinstance(questions, str):
        questions = [questions]

    where_clause = {"source": source_filter} if source_filter else None

    if method == 'vector':
        batch_results = vault.query(questions, n_results=n_results, where=where_clause)
    elif method == 'hybrid':
        results = vault.hybrid_query(questions, n_results=n_results, where=where_clause)
    else: # Default to rerank
        batch_results = vault.rerank_query(questions, n_results=n_results, where=where_clause)

    all_messages = []
    for q_idx, question in enumerate(questions):
        results = batch_results[q_idx]
        context_parts = []

        if show_context:
            print(f"\n--- LIBRARIAN REPORT: Question {q_idx+1} ({len(results)} chunks via {method.upper()}) ---")

        for i, res in enumerate(results):
            text = res['text']
            meta = res['metadata']

            if show_context:
                print(f"Chunk {i+1}: Source: {meta['source']}, Page: {meta['page']} (ID: {res['id']})")
                print(f"Preview: {text[:80]}...\n")

            context_parts.append(f"[SOURCE: {meta['source']}, PAGE: {meta['page']}]\n{text}")

        context = "\n---\n".join(context_parts)

        # Build the message list for this specific question
        messages = [
            {"role": "system", "content": "You are a Senior FRA Safety Consultant. Use your 4-Phase Thinking Process. Answer ONLY based on the provided context."},
            {"role": "user", "content": f"CONTEXT FROM MANUALS:\n{context}\n\nQUESTION:\n{question}"},
        ]
        all_messages.append(messages)

    # # Extract text and metadata for the prompt
    # context_parts = []
    # for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    #     context_parts.append(f"[SOURCE: {meta['source']}, PAGE: {meta['page']}]\n{doc}")
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in all_messages]

    # Ensure pad_token is set (crucial for batching)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            use_cache=True,
            temperature=0,
            do_sample=False
        )

    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return response#[0].split("assistant")[-1].strip()


In [ ]:
# query_test = vault.query(question = "A rail yard manager wants to move a leaking LPG tank car to a remote siding for repair without a permit. Is this allowed?", n_results = 1)
# query_test
search_params = {"n_results": 1}
source_filter = "FRA-Hazardous_Materials_Compliance_Manual_01.07.25.pdf"
search_params['where'] = {"source": source_filter}
question = "A rail yard manager wants to move a leaking LPG tank car to a remote siding for repair without a permit. Is this allowed?"
query_vector = vault.embedder.generate_embeddings([question])
query_list = query_vector.tolist() if hasattr(query_vector, 'tolist') else query_vector
query_test = vault.collection.query(
        query_embeddings=query_list,
        **search_params
    )
query_test

In [ ]:
hazmat_file = "FRA-Hazardous_Materials_Compliance_Manual_01.07.25.pdf"
test_question = "A rail yard manager wants to move a leaking LPG tank car to a remote siding for repair without a permit. Is this allowed?"
final_answer = run_integrated_audit(test_question, vault, n_results = 5, source_filter = hazmat_file, show_context = True)
print(final_answer[0].split("assistant")[-1].strip())

In [ ]:
final_answer_no_soruce = run_integrated_audit(test_question, vault, n_results = 5, show_context = True)
print(final_answer_no_soruce[0].split("assistant")[-1].strip())

In [ ]:
dummy = vault.collection.get(include = ['documents', 'metadatas'])

In [ ]:
try:
    index = dummy['ids'].index('id_2304')
    print(f"The index of 'id_2304' is: {index}")
    print(dummy['documents'][index])
    print(dummy['metadatas'][index])
except ValueError:
    print("'id_2304' not found in dummy['ids']")

# try:
#     index = dummy['ids'].index('id_572')
#     print(f"The index of 'id_572' is: {index}")
#     print(dummy['documents'][index])
#     print(dummy['metadatas'][index])
# except ValueError:
#     print("'id_572' not found in dummy['ids']")